# The joint arm: one encoder that keeps the content and is shaped for dynamics
What the B experiments taught (`docs/findings.md`): the **shadow** (reconstruction + SIGReg) holds the content (hand, agent, facing, inventory at
decoder strength 0.9-1.0) but its latent is **not shaped for dynamics**: its predictor barely foresees the facing after a move (decoded accuracy
0.36-0.46 against 0.96 for the real next frame; copying the last frame gives 0.28), and no predictor variant we tried on that frozen latent got far
(best 0.59). The **live** encoder (prediction + SIGReg + retention) has a latent about 5x more predictable but loses hard content unless retention
is strong enough. B splits the two properties across two encoders joined by a linear readout.

The **joint arm** puts both into one encoder: class-weighted reconstruction (keeps the content) + the prediction gradient through the context path
(shapes the latent so the dynamics are simple) + SIGReg. No shadow, no retention, no gate.

| run | what it is |
|---|---|
| **shadow alone (`ae`)** | reconstruction + SIGReg, the predictor learns on detached latents (the B shadow on its own, now at 15k so the facing transition at 8-12k is reached) |
| **joint** | the same + the prediction gradient into the encoder |
| **joint + FiLM predictor** | the same with the residual, action-gated predictor (learned the facing faster in a fork: 0.46 vs 0.27) |

All three: D = 128, SIGReg 0.01, class weights (power 1), 15k steps, seed 0, the arrow agent and chasing mob. Every result is read **with a decoder**
(`gaf/readout.py`): the probes under-read compressed latents. The B reference (best run, open path on + floor) is read from `/workspace/results_mech`
if it is still there.

In [ ]:
%pip install -q -e .
import torch
print(torch.__version__, torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NO GPU (CPU fallback, slow)")

In [ ]:
%matplotlib inline
from pathlib import Path
import numpy as np
import pandas as pd
from IPython.display import display
from gaf.env_grid import GridCfg
from gaf.train import Cfg
from gaf import analysis as A
from gaf import readout as R

OUT = "/workspace/results_joint"
B_REF = Path("/workspace/results_mech/retain_open_s0_386e3e97")   # B, open path on + floor on all (best B so far); optional
SEED, D, STEPS, WORKERS = 0, 128, 15_000, 3
ENV = GridCfg(W=12, H=8, hand_tiles=1, agent_shape="arrow", mob_mode="chase", n_mobs=1, mob_chase=0.7)
COMMON = dict(lam=0.01, batch=256, lr=1e-3, wd=0.01, steps=STEPS, floor_start=10 ** 9, eval_every=1000, eval_parts=12_000,
              eval_parts_steps=1000, D=D, seed=SEED, env=ENV, rec_class_power=1.0, ckpt_every=5000, log_mech=3000)
RUNS = {"shadow alone (ae)": Cfg(arm="ae", tag="joint-ae", **COMMON),
        "joint": Cfg(arm="joint", tag="joint", **COMMON),
        "joint + FiLM predictor": Cfg(arm="joint", tag="joint-film", pred_arch="film", **COMMON)}
pd.set_option("display.precision", 3)

## Launch
Background process (safe to re-run: finished and running runs are skipped). To stop: `kill -TERM -- -$(cat /workspace/results_joint/launch_*.pid)`.

In [ ]:
A.launch(list(RUNS.values()), OUT, workers=WORKERS)

In [ ]:
display(A.progress(OUT, list(RUNS.values())))

In [ ]:
A.live_probes(OUT, RUNS, every=30)

**Recovery cell.** Only if a run died in its final probes (it has `ckpt_<steps>.pt` and a full `metrics.csv` but no `summary.json`).

In [ ]:
for name, c in RUNS.items():
    d = Path(OUT) / c.run_id
    if d.exists() and not (d / "summary.json").exists() and (d / f"ckpt_{c.steps}.pt").exists():
        print("finalizing", name)
        A.finalize(c, OUT)

## Dynamics: what each model foresees
Each model's own decoder applied to the real next latent (ceiling), to the predictor's output (what the model foresees) and to the last frame
(floor). **The column to read first is `facing acc, move steps`**: on steps where the action turned or moved the agent, copying the last frame is
mostly wrong and the ceiling is about 0.96. For B only the shadow has a decoder (its live row shows only the latent error).

In [ ]:
dyn = {name: R.dynamics(Path(OUT) / c.run_id) for name, c in RUNS.items() if (Path(OUT) / c.run_id / "summary.json").exists()}
if (B_REF / "summary.json").exists():
    dyn["B (open on, floor all)"] = R.dynamics(B_REF)
DYN = pd.concat(dyn)
display(DYN.round(2))

## Content: what each latent holds, at decoder strength
A fresh class-weighted decoder on each frozen latent (5000 steps, about 1-2 minutes each). Terrain and mob are under-read by the class weights.
Reference (measured before, same readout): B's live encoder held hand 0.94, agent 0.87, facing 0.83, twinkle 0.47; its shadow hand 1.00,
agent 0.90, facing 0.93, twinkle 0.67.

In [ ]:
content = {name: R.decoder_strength(Path(OUT) / c.run_id) for name, c in RUNS.items() if (Path(OUT) / c.run_id / "summary.json").exists()}
CONTENT = pd.concat(content)
display(CONTENT.round(2))

## Verdicts
Rules written before the results.
* **Content kept.** The joint encoder holds hand, agent and facing at decoder strength >= 0.8 (as the shadow does): reconstruction keeps the content
  in an encoder that also predicts.
* **Dynamics learned.** The joint model's predicted frame reaches move-step facing >= 0.8 and agent >= 0.8, clearly above the shadow alone
  (whose predictor learns on detached latents) and above B's shadow (0.36-0.46): the prediction gradient shapes the latent for dynamics.
* **Hand dynamics.** Predicted hand more than 0.3 above copy-last: the hand's dynamics are learnable here (the terrain next to the agent is in the
  latent). If not, the terrain is missing (the class weights push it to the cap): the next lever is a softer weight cap.
* **Noise.** The joint arm has no noise filter: twinkle is expected to be kept. That is the open part of the recipe.
* **Verdict.** If the joint arm matches B on content and beats it on dynamics, the single-encoder recipe replaces B, and what is left to add is a
  noise criterion (independence from history and action) and mob handling.

In [ ]:
def get(df, idx, col):
    try:
        return float(df.loc[idx, col])
    except (KeyError, TypeError):
        return float("nan")


def say(label, ok):
    print(f"{label}: {'yes' if ok else 'no'}")


for name in RUNS:
    if name not in dyn:
        print(name, "not finished"); continue
    print(f"--- {name}")
    if "CONTENT" in globals() and name in content:
        hold = {p: get(content[name], "live", p) for p in ("hand", "agent", "facing acc")}
        print("  content (decoder strength):", {k: round(v, 2) for k, v in hold.items()})
        say("  content kept (hand, agent, facing >= 0.8)", all(v >= 0.8 for v in hold.values()))
    pf, pa = get(dyn[name], ("live", "predicted"), "facing acc, move steps"), get(dyn[name], ("live", "predicted"), "agent")
    ph, ch = get(dyn[name], ("live", "predicted"), "hand"), get(dyn[name], ("live", "copy last frame"), "hand")
    print(f"  predicted: facing (move steps) {pf:.2f}, agent {pa:.2f}, hand {ph:.2f} (copy last {ch:.2f})")
    say("  dynamics learned (facing, agent >= 0.8)", pf >= 0.8 and pa >= 0.8)
    say("  hand dynamics learned (> copy last + 0.3)", ph > ch + 0.3)

## Final readout
The strong final probes of every run (probes under-read; judge content by the decoder tables above).

In [ ]:
df = A.collect(OUT)
rows = {}
for name, c in RUNS.items():
    r = df[df.run_id == c.run_id]
    if len(r):
        rows[name] = {**{p: r[f"pm_{p}"].mean() for p in ["hand", "position", "inventory", "facing", "mob", "twinkle"]}, "rank": r["pm_rank"].mean()}
display(pd.DataFrame(rows).T.round(3))